_ExplainDB — Copyright (C) 2026 Prof. Dr. Jens Dittrich, Saarland University._  
_Licensed under the GNU Affero General Public License v3 (AGPL-3.0). See the `LICENSE` file._

In [ ]:
import sys, pathlib

# Put the repo root (the directory containing `system/`) on sys.path so
# `from system...` works whether this notebook is opened from its own
# folder or from the repo root.
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "system").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

**Data Layouts POC**

This notebook shows the effects of switching from row to column layout. For educational purposes we use csv-files. Note that in a real database, we would use a binary format. Note that these effects do not only happen when writing data to files (**serializing**) but at any time when we **linearize** data, i.e. every time we decide how to map higher-dimensional data to a one-dimensional data space.

**Create some fake data**

In [1]:
from system.data_classes import *
from faker import Faker

Faker.seed(42)
fake: Faker = Faker()

number_of_tuples: int = 10000

# create some fake data mimicking tuples in a relation
books: list[Book] = [
    Book(
        fake.sentence(),
        fake.name(),
        fake.pyfloat(min_value=0, max_value=100, right_digits=2),
    )
    for i in range(number_of_tuples)
]
books[:10]

[Book(title='Agent every development say.', author='Stephanie Miller', price=54.4),
 Book(title='Behavior discussion own.', author='Michelle Miles', price=53.28),
 Book(title='Operation speak according south recently future choice.', author='Gabrielle Davis', price=12.45),
 Book(title='Page southern role movie win her.', author='Sandra Montgomery', price=46.73),
 Book(title='Relate animal direction eye bag.', author='Brian Ramirez', price=81.46),
 Book(title='Have decide environment view.', author='Lisa Jackson', price=93.31),
 Book(title='Left establish understand read.', author='Frederick Tate', price=7.29),
 Book(title='Source husband at.', author='Latoya Robbins', price=27.83),
 Book(title='How trip learn enter east no enjoy.', author='Stephanie Ross', price=28.17),
 Book(title='Management sense technology check civil quite others.', author='Kyle Mcdonald', price=32.7)]

In [2]:
from pandas import DataFrame, read_csv

# store data in pandas dataframe:
schema: list[str] = ["title", "author", "price"]
df: DataFrame = DataFrame(data=books, columns=schema)
df

,title,author,price
0,Agent every development say.,Stephanie Miller,54.40
1,Behavior discussion own.,Michelle Miles,53.28
2,Operation speak according south recently futur...,Gabrielle Davis,12.45
3,Page southern role movie win her.,Sandra Montgomery,46.73
4,Relate animal direction eye bag.,Brian Ramirez,81.46
...,...,...,...
9995,Remember result act.,Mark Clark,29.10
9996,Tonight list Democrat year forward.,Danny Walker,57.94
9997,President star room partner.,Jennifer Chavez,0.26
9998,Well bag develop support.,Katherine Ramos,6.50


**Emulate row store**

To emulate a row-major layout, we write the data into a CSV file where each tuple is stored consecutively. Within each tuple, each attribute is also stored consecutively. Inspect the generated `row.csv` file for more detail.

In [3]:
# write data into file in row-major layout to emulate store
path_row: str = "row.csv"
df.to_csv(path_or_buf=path_row, sep="\n", header=False, index=False, mode="w")

In [4]:
!head row.csv

Agent every development say.
Stephanie Miller
54.4
Behavior discussion own.
Michelle Miles
53.28
Operation speak according south recently future choice.
Gabrielle Davis
12.45
Page southern role movie win her.


**Emulate column store**

To emulate a column-major layout, we write the data into a CSV file where each attribute column is stored consecutively. Inspect the generated `column.csv` file for more detail.

In [5]:
# write data into file in column-major layout to emulate store
path_column: str = "column.csv"
attribute_index: int
col: str
for attribute_index, col in enumerate(df.columns):
    df.to_csv(
        path_or_buf=path_column,
        columns=[col],
        header=False,
        index=False,
        mode=(
            "w" if attribute_index == 0 else "a"
        ),  # truncate file before first column, append afterward
    )

In [6]:
!head column.csv

Agent every development say.
Behavior discussion own.
Operation speak according south recently future choice.
Page southern role movie win her.
Relate animal direction eye bag.
Have decide environment view.
Left establish understand read.
Source husband at.
How trip learn enter east no enjoy.
Management sense technology check civil quite others.


**Consider the following query accessing multiple attributes**
``` sql
SELECT author, price 
FROM books
LIMIT 314
OFFSET 100
```

This query prints a subset of tuples (only 314 tuple skipping the first 100 tuples) and a subset of attributes (only author and price). However, for each print call, we need multiple attribute values of the *same* tuple. In theory, we expect the row-major layout to be superior as these attribute values are stored consecutively in the row store. So let's measure the actual query execution times! 

Note that our query engine basically loads the needed values again in a pandas dataframe in the given layout. Afterward, the values forming a single tuple has to be accessed (consecutive accesses for row-major layout, random accesses for column-major layout). Each constructed tuple is then printed.

In [7]:
from time import time_ns

attr_idxs: list[int] = [1, 2]

# start row:
offset: int = 100

# number of rows to consider:
limit: int = 314

row_start: int = time_ns()

# execute query given row-major layout:
data: DataFrame = read_csv(
    filepath_or_buffer=path_row,
    header=None,
    skiprows=lambda line_number: line_number
    < offset * len(schema)  # before offset, computes which rows to skip from the input
    or line_number % len(schema)
    not in attr_idxs,  # wrong attribute: we are after the offset but only want to read the attribute values projected to
    nrows=limit * len(attr_idxs),  # do not exceed limit
)
i: int
for i in range(0, len(attr_idxs) * limit, len(attr_idxs)):
    # construct tuple by consecutive accesses
    l = [data.iloc[i + j][0] for j in range(len(attr_idxs))]
    # print(l, sep=", ")

row_end: int = time_ns()

print("Time row-major:", round((row_end - row_start) / 1000000, 1), "ms")

Time row-major: 9.6 ms


In [8]:
column_start: int = time_ns()

# execute query given column-major layout
data: DataFrame = read_csv(
    filepath_or_buffer=path_column,
    header=None,
    skiprows=lambda line_number: line_number % number_of_tuples
    < offset  # before offset
    or line_number % number_of_tuples >= offset + limit  # after limit in column
    or line_number // number_of_tuples not in attr_idxs,  # wrong column index
    nrows=limit * len(attr_idxs),  # break after this many entries have been produced
)
i: int
for i in range(limit):
    # construct tuple by random accesses:
    l = [data.iloc[j * limit + i][0] for j in range(len(attr_idxs))]
    # print(l, sep=", ")

column_end: int = time_ns()

print("Time column-major:", round((column_end - column_start) / 1000000, 1), "ms")

Time column-major: 10.9 ms


We can observe that the row-major layout indeed favors the given query accessing multiple attributes.

**Consider the following query accessing and aggregating a single attribute**
``` sql
SELECT MAX(price) 
FROM books
```

This query aggregates over a single attribute (price). Therefore, we now expect the column-major layout to be superior as all attribute values are stored consecutively in the column store. So again let's measure the actual query execution times! 

Note that our query engine still loads the needed values in a pandas dataframe (random accesses for row-major layout, consecutive accesses for column-major layout). Afterward, all loaded values are aggregated and the result is then printed.

In [9]:
attr_idx: int = 2

row_start: int = time_ns()

# execute query given row-major layout
data: DataFrame = read_csv(
    filepath_or_buffer=path_row,
    header=None,
    skiprows=lambda x: x % len(schema) != attr_idx,  # wrong attribute
    nrows=number_of_tuples,
)
aggregate: float = data.max(0)[0]  # compute aggregate
row_end: int = time_ns()

print("Aggregate:", aggregate)

print("Time row-major:", round((row_end - row_start) / 1000000, 1), "ms")

Aggregate: 99.98
Time row-major: 5.9 ms


In [10]:
column_start: int = time_ns()

# execute query given column-major layout
data: DataFrame = read_csv(
    filepath_or_buffer=path_column,
    header=None,
    skiprows=attr_idx * number_of_tuples,  # start at attribute column
    nrows=number_of_tuples,
)
aggregate: float = data.max(0)[0]  # compute aggregate
column_end: int = time_ns()

print("Aggregate:", aggregate)

print("Time column-major:", round((column_end - column_start) / 1000000, 1), "ms")

Aggregate: 99.98
Time column-major: 3.3 ms


In [11]:
!rm row.csv column.csv

We can observe that now the column-major layout indeed favors the given query accessing and aggregating a single attribute.  Note that we could additionally speed up the execution using the column-major layout by utilizing vectorization and SIMD instructions.